# 01 — Data Cleaning
## Global Job Market Compensation Analysis

**Objective:** Apply only the cleaning steps justified by the Phase 2 audit — no template padding.

**Phase 2 audit findings driving this notebook:**
- 500,000 rows, 12 columns, **zero missing values** across all columns
- **28 exact duplicate rows** (0.006%)
- Categorical labels are already clean and consistent (verified below, not assumed)
- **No global salary outliers** (IQR) — but global IQR can mask group-level anomalies, so we re-check within `country` x `occupation`
- Dataset shows signs of synthetic generation (perfectly balanced categorical distributions, zero nulls) — this is documented here and in the project README, not hidden

**Policy for this notebook:** flag anomalies, never silently auto-delete. Every removal is logged with a before/after row count.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', None)

RAW_PATH = r"C:\Users\rashm\Downloads\Job Market\data\raw\job_market_raw.csv"
df = pd.read_csv(RAW_PATH)

print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

Loaded: 500,000 rows, 12 columns


,country,city,occupation,field,years_of_experience,salary,employment_type,education_level,gender,company_size,year,month
0,Switzerland,Zurich,Operations Manager,Operations,16,359609,work_from_home,Master,Male,Large,2023,8
1,India,Bangalore,HR Analyst,Human Resources,12,79059,part_time,PhD,Male,Large,2023,5
2,Sweden,Stockholm,Software Engineer,Technology,10,258077,freelance,Master,Male,Enterprise,2023,9
3,South Korea,Seoul,Operations Manager,Operations,7,252282,part_time,Master,Female,Large,2024,12
4,United States,New York,Cloud Engineer,Technology,4,330618,part_time,Master,Male,Enterprise,2022,1


## 1. Duplicate Removal

The audit found 28 exact full-row duplicates. We drop them, but log the before/after count -- silent drops are never acceptable in a reproducible pipeline.

In [2]:
rows_before = len(df)
dupe_count = df.duplicated().sum()

df = df.drop_duplicates().reset_index(drop=True)

rows_after = len(df)
print(f"Duplicates found : {dupe_count}")
print(f"Rows before      : {rows_before:,}")
print(f"Rows after       : {rows_after:,}")
print(f"Rows removed     : {rows_before - rows_after} ({(rows_before-rows_after)/rows_before:.4%})")

assert rows_before - rows_after == dupe_count, "Mismatch between detected and dropped duplicates"

Duplicates found : 28
Rows before      : 500,000
Rows after       : 499,972
Rows removed     : 28 (0.0056%)


## 2. Categorical Standardization Check ("trust but verify")

The audit's `.value_counts()` output showed no obvious typos or case inconsistencies. Rather than assume that holds after deduplication, we run a defensive check: strip whitespace, check case consistency, and confirm the label sets are unchanged. This costs almost nothing to run and converts an assumption into a verified fact.

In [3]:
categorical_cols = ['country', 'city', 'occupation', 'field', 'employment_type',
                    'education_level', 'gender', 'company_size']

standardization_report = {}
for col in categorical_cols:
    before_labels = set(df[col].unique())
    stripped = df[col].astype(str).str.strip()
    case_variants = stripped.str.lower().nunique() != stripped.nunique()
    # This checks whether removing whitespace changed any values.
    whitespace_issues = (df[col].astype(str) != stripped).sum()
    standardization_report[col] = {
        'n_unique': len(before_labels),
        'whitespace_issues': whitespace_issues,
        'case_inconsistency_detected': case_variants
    }
    df[col] = stripped

report_df = pd.DataFrame(standardization_report).T
report_df


,n_unique,whitespace_issues,case_inconsistency_detected
country,21,0,False
city,22,0,False
occupation,12,0,False
field,7,0,False
employment_type,5,0,False
education_level,4,0,False
gender,3,0,False
company_size,4,0,False


## 3. Data Type Conversion

Convert low-cardinality string columns to `category` dtype. This is not cosmetic -- it cuts memory footprint substantially and is handled more efficiently and correctly by scikit-learn, XGBoost, and statsmodels than raw `object` columns.

In [4]:
mem_before = df.memory_usage(deep=True).sum() / 1e6

for col in categorical_cols:
    df[col] = df[col].astype('category')

mem_after = df.memory_usage(deep=True).sum() / 1e6

print(f"Memory before category conversion: {mem_before:.1f} MB")
print(f"Memory after category conversion : {mem_after:.1f} MB")
print(f"Reduction                        : {(1 - mem_after/mem_before):.1%}")

df.dtypes


Memory before category conversion: 83.1 MB
Memory after category conversion : 20.0 MB
Reduction                        : 75.9%


country                category
city                   category
occupation             category
field                  category
years_of_experience       int64
salary                    int64
employment_type        category
education_level        category
gender                 category
company_size           category
year                      int64
month                     int64
dtype: object

## 4. Logical Consistency Assertions

Rather than eyeball plausibility, we assert it in code so these checks are testable and rerun automatically if the pipeline changes upstream.

In [5]:
assert df['year'].between(2022, 2025).all(), "year out of expected range"
assert df['month'].between(1, 12).all(), "month out of 1-12 range"
assert df['salary'].gt(0).all(), "non-positive salary found"
assert df['years_of_experience'].between(0, 25).all(), "years_of_experience out of expected range"

mapping_check = df.groupby('occupation', observed=True)['field'].nunique()
assert (mapping_check == 1).all(), "occupation now maps to multiple fields -- investigate"

print("All logical consistency checks passed.")


All logical consistency checks passed.


## 5. Group-Level Outlier Check (salary within country x occupation)

Global IQR found no salary outliers, but that can mask anomalies that only look extreme *within* a specific country/occupation combination. We compute IQR bounds per group and **flag** (not delete) any rows outside them, per our standing policy.

In [6]:
    def flag_group_outliers(group):
        q1, q3 = group['salary'].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        return (group['salary'] < lower) | (group['salary'] > upper)
    
    df['salary_outlier_flag'] = (
        df.groupby(['country', 'occupation'], observed=True)
          .apply(flag_group_outliers, include_groups=False)
          .reset_index(level=[0, 1], drop=True)
          .sort_index()
    )
    
    n_flagged = df['salary_outlier_flag'].sum()
    print(f"Group-level (country x occupation) salary outliers flagged: {n_flagged:,} "
          f"({n_flagged/len(df):.3%} of rows)")
    
    df[df['salary_outlier_flag']][['country','occupation','years_of_experience','salary','education_level']].head(10)


Group-level (country x occupation) salary outliers flagged: 10,435 (2.087% of rows)


,country,occupation,years_of_experience,salary,education_level
27,Singapore,HR Analyst,1,13576,High School
102,South Africa,Product Manager,17,334423,High School
117,India,AI Engineer,8,257698,Bachelor
404,Spain,Data Analyst,0,22317,Bachelor
482,United Arab Emirates,HR Analyst,0,20965,High School
495,New Zealand,UX Designer,9,370000,Master
516,Canada,Software Engineer,0,20253,PhD
648,Brazil,Business Analyst,20,298509,PhD
828,Sweden,HR Analyst,1,22382,High School
884,Germany,Marketing Specialist,12,344466,PhD


**Decision:** these rows are flagged, not removed. They stay in the main dataset for EDA/modeling; anyone downstream can filter on `salary_outlier_flag` if they want an outlier-free view. We will revisit this flag qualitatively in Phase 5 EDA -- if flagged rows turn out to be concentrated in one country/occupation pair, that's itself a finding worth reporting, not noise to discard.

## 6. Data Provenance Note (carried into README)

This dataset shows strong indicators of synthetic generation:
- Zero missing values across all 500,000 rows and 12 columns
- Near-perfectly balanced categorical distributions (e.g., education levels ~125K each, employment types ~100K each)
- No global salary outliers despite a wide $12K-$370K range

This is documented here explicitly rather than presented as organically messy real-world data. It does not invalidate the analysis -- it means our statistical tests will behave cleanly (balanced groups, no missingness handling needed) -- but it does mean we should avoid over-interpreting subtle distributional quirks as "real" labor market signal, since some of the structure is a generation artifact.

**Currency assumption (explicit, not verifiable from data):** `salary` is treated as already USD-normalized across all countries. No currency or FX field exists in the source data, so this is a documented assumption, not a verified fact, and every cross-country salary comparison in this project carries that caveat.

## 7. Save Cleaned Dataset

In [7]:
from pathlib import Path

# Output directory
OUTPUT_DIR = Path(r"C:\Users\rashm\Downloads\Job Market\data\processed")

# Create the folder if it doesn't already exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output file path
OUT_PATH = OUTPUT_DIR / "job_market_cleaned.parquet"

# Delete the existing file if it exists.
if OUT_PATH.exists():
    OUT_PATH.unlink()

# Save cleaned dataset
df.to_parquet(OUT_PATH, index=False)

print("=" * 60)
print("Cleaned dataset saved successfully")
print("=" * 60)
print(f"Location    : {OUT_PATH}")
print(f"Rows        : {df.shape[0]:,}")
print(f"Columns     : {df.shape[1]}")
print(f"File Format : Parquet")
print("=" * 60)

df.info()

Cleaned dataset saved successfully
Location    : C:\Users\rashm\Downloads\Job Market\data\processed\job_market_cleaned.parquet
Rows        : 499,972
Columns     : 13
File Format : Parquet
<class 'pandas.DataFrame'>
RangeIndex: 499972 entries, 0 to 499971
Data columns (total 13 columns):
 #   Column               Non-Null Count   Dtype   
---  ------               --------------   -----   
 0   country              499972 non-null  category
 1   city                 499972 non-null  category
 2   occupation           499972 non-null  category
 3   field                499972 non-null  category
 4   years_of_experience  499972 non-null  int64   
 5   salary               499972 non-null  int64   
 6   employment_type      499972 non-null  category
 7   education_level      499972 non-null  category
 8   gender               499972 non-null  category
 9   company_size         499972 non-null  category
 10  year                 499972 non-null  int64   
 11  month                499972 non